In [2]:
# ============================================
# BLACK HOLE SIMULATION - LIVE ANIMATION
# ============================================

%matplotlib osx

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def simulate_particle(x0, y0, vx0, vy0, steps=18000, dt=1):
    r = np.sqrt(x0**2 + y0**2)
    phi = np.arctan2(y0, x0)
    vr   = (x0 * vx0 + y0 * vy0) / r
    vphi = (x0 * vy0 - y0 * vx0) / r**2
    rs_norm = 1.0
    M_norm  = 0.5
    factor = max(1.0 - (rs_norm / r), 0.001)
    L = r**2 * vphi
    V_eff_init = factor * (1.0 + (L**2 / r**2))
    E = factor * (1.0 + abs(vr))
    if E**2 < V_eff_init:
        E = np.sqrt(V_eff_init) * 1.05

    path_x, path_y, path_r = [], [], []

    for _ in range(steps):
        r_acc = (
            - M_norm / r**2
            + L**2   / r**3
            - 3 * M_norm * L**2 / r**4
        )
        phi_dot = L / r**2
        vr  += r_acc  * dt
        r   += vr     * dt
        phi += phi_dot * dt

        if r <= rs_norm * 1.05:
            print(f"  → Absorbed at step {_}")
            break
        if r > 65 or r <= 0.01:
            print(f"  → Escaped at step {_}")
            break

        path_x.append(r * np.cos(phi))
        path_y.append(r * np.sin(phi))
        path_r.append(r)

    return np.array(path_x), np.array(path_y), np.array(path_r)

# ============================================
# PARTICLES (Initial Conditions)
# ============================================

bh_particles = [
    ( 22,   0,  -0.15,   0.15,  'red',     'P1 - Test1'),
    ( 65,   0,    0,   0,  'orange',  'P2 - Test2'),
    ( 10,   0,    0.01,   0.00,  'yellow',  'P3 - Test3'),
    ( 20,   0,    0,   0.16,  'hotpink', 'P4 -  Test4'),
    ( 17,   0,    0.0,   0.2,  'violet',  'P5 - Test5'),
]

# ============================================
# COMPUTE PATHS
# ============================================

print("Computing particle paths...")
bh_paths = []
for i, p in enumerate(bh_particles):
    print(f"  Simulating {p[5]}...")
    path = simulate_particle(*p[:4])
    bh_paths.append(path)
    print(f"  Path length: {len(path[0])} steps")
print("All paths computed!")

# ============================================
# SETUP FIGURE
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                         gridspec_kw={'width_ratios': [3, 1]})
fig.patch.set_facecolor('black')

ax     = axes[0]
ax_bar = axes[1]

# --- Main simulation axis ---
ax.set_facecolor('black')
ax.set_xlim(-55, 55)
ax.set_ylim(-50, 50)
ax.set_aspect('equal')
ax.set_title('Orbital Stability and Capture Dynamics Near a Black Hole', color='white', fontsize=13)
ax.set_xlabel('X (rs)', color='white')
ax.set_ylabel('Y (rs)', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.1, color='white')

# Event horizon
ax.add_patch(Circle((0,0), 5.0, color='red', fill=False,
                    linewidth=2, linestyle='--', label='Event horizon'))

# Singularity
ax.add_patch(Circle((0,0), 1.0, color='white', fill=True))

# Glow
for rad, alp in [(8,0.04),(6,0.08),(5.2,0.14)]:
    ax.add_patch(Circle((0,0), rad, color='orange', alpha=alp))

ax.legend(facecolor='black', labelcolor='white', loc='upper right', fontsize=9)

gravity_dot, = ax_bar.plot([], [], 'o', color='white', ms=10, zorder=5)
gravity_text = ax_bar.text(0.02, 0.5, '', color='white', fontsize=8)

# ============================================
# ANIMATED ELEMENTS
# ============================================

bh_trails = [ax.plot([], [], '-', color=p[4], lw=1.5, alpha=0.85)[0] for p in bh_particles]
bh_dots   = [ax.plot([], [], 'o', color=p[4], ms=8)[0]               for p in bh_particles]
bh_labels = [ax.text(0, 0, p[5], color=p[4], fontsize=8)             for p in bh_particles]

plt.tight_layout()
plt.ion()
plt.show()

# ============================================
# ANIMATION LOOP
# ============================================

trail_length = 100000
total_frames = 300

print("Animation running...")

for frame in range(total_frames):
    closest_r = 999

    for path, trail, dot, label in zip(bh_paths, bh_trails, bh_dots, bh_labels):
        px, py, pr = path
        n = len(px)
        if n < 2:
            continue
        idx   = min(int((frame / total_frames) * n), n - 1)
        start = max(0, idx - trail_length)
        trail.set_data(px[start:idx], py[start:idx])
        dot.set_data([px[idx]], [py[idx]])
        label.set_position((px[idx] + 0.8, py[idx] + 0.8))

        if pr[idx] < closest_r:
            closest_r = pr[idx]

    if closest_r < 999:
        g_strength = min(1.0 / closest_r**2, 1.0)
        g_norm = min(g_strength / (1.0 / 0.5**2), 1.0)
        gravity_dot.set_data([g_norm], [closest_r])
        gravity_text.set_text(
            f'Closest: {closest_r:.1f} rs\n'
            f'Gravity: {g_strength:.3f}'
        )

    fig.canvas.draw()
    fig.canvas.flush_events()
    plt.pause(0.05)

plt.ioff()
print("Animation complete!")

Computing particle paths...
  Simulating P1 - Test1...
  → Escaped at step 547
  Path length: 547 steps
  Simulating P2 - Test2...
  → Absorbed at step 821
  Path length: 821 steps
  Simulating P3 - Test3...
  → Absorbed at step 50
  Path length: 50 steps
  Simulating P4 -  Test4...
  Path length: 18000 steps
  Simulating P5 - Test5...
  Path length: 18000 steps
All paths computed!
Animation running...
Animation complete!
